# Embeddings en Colab (Fase 3)

Calcula embeddings de `prompt`, `response_a` y `response_b` con `BAAI/bge-small-en-v1.5`
(384 dimensiones) para train (y para test si el archivo existe), y los guarda en Google Drive. Se ejecuta en Colab con
GPU (Entorno de ejecución > Cambiar tipo > T4) porque en CPU tardaría horas.

Antes de correrlo, `train.csv` debe estar en la raíz de "Mi unidad". Los embeddings se guardan en
`MyDrive/llm-classification/`.
Cada texto único se codifica una sola vez.

In [ ]:
!pip -q install -U sentence-transformers
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

DRIVE = Path("/content/drive/MyDrive")
OUT = DRIVE / "llm-classification"
OUT.mkdir(exist_ok=True)
TRAIN_ROWS = 57_477
MODEL = "BAAI/bge-small-en-v1.5"
MAX_LEN = 512

assert torch.cuda.is_available(), "Activa una GPU: Entorno de ejecución > Cambiar tipo"
model = SentenceTransformer(MODEL, device="cuda")
model.max_seq_length = MAX_LEN
model.half()
print(torch.cuda.get_device_name(0))

In [ ]:
def parse(raw):
    # Same parsing as src/data.py: JSON list of turns, null -> "", turns joined by a blank line.
    return "\n\n".join("" if t is None else t for t in json.loads(raw))


def embed_split(name, expected_rows=None):
    path = DRIVE / f"{name}.csv"
    if not path.exists():
        print(f"{path} not found, skipping")
        return
    df = pd.read_csv(path, usecols=["id", "prompt", "response_a", "response_b"])
    if expected_rows:
        assert len(df) == expected_rows, f"{name}.csv has {len(df)} rows, expected {expected_rows} (incomplete upload?)"
    for col in ["prompt", "response_a", "response_b"]:
        df[col] = df[col].map(parse)

    texts = pd.unique(pd.concat([df["prompt"], df["response_a"], df["response_b"]]))
    print(f"{name}: {len(df):,} rows, {len(texts):,} unique texts")
    vectors = model.encode(list(texts), batch_size=128, normalize_embeddings=True,
                           show_progress_bar=True, convert_to_numpy=True).astype(np.float16)

    index = {t: i for i, t in enumerate(texts)}
    out = {col: vectors[df[col].map(index).to_numpy()] for col in ["prompt", "response_a", "response_b"]}
    np.savez_compressed(OUT / f"emb_{name}.npz", id=df["id"].to_numpy(), **out)
    print("saved", OUT / f"emb_{name}.npz")

In [ ]:
embed_split("test")   # skipped if test.csv is not in Drive; the real test set is embedded in the Kaggle notebook

In [ ]:
embed_split("train", TRAIN_ROWS)

In [ ]:
z = np.load(OUT / "emb_train.npz")
print({k: z[k].shape for k in z.files})
cos = (z["response_a"].astype(np.float32) * z["response_b"].astype(np.float32)).sum(1)
print("cosine(A, B): mean", cos.mean().round(3), "min", cos.min().round(3))